# PySpark Vodafone Analysis

This notebook processes credit card transaction data to count transactions per card.

In [10]:
import findspark
findspark.init()

**Import Spark modules** - Import SparkSession and SparkConf for DataFrame operations.

In [11]:
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf

**Create SparkSession** - Configure Spark to run in local mode using all available CPU cores (`local[*]`).

In [12]:
spark = SparkSession.builder\
    .master("local[*]")\
    .appName("WordCount")\
    .getOrCreate()

**Get SparkContext** - Access the underlying RDD API through the SparkContext.

In [13]:
sc = spark.sparkContext

**Load Dataset** - Read the input CSV file as an RDD. Each line becomes one element in the RDD.

In [14]:
dataset_path="in.csv"
cards_rdd=sc.textFile(dataset_path)


**Preview Data** - Display the first 5 lines of the raw RDD to understand the data format.

In [15]:
cards_rdd.take(5)

['224,10', '836,100', '81,5', '809,50', '786,25']

**Transform Data** - Parse each line by splitting on comma and convert to integers. Creates pairs of (card_id, transaction_amount).

In [30]:
cards_transactions = cards_rdd.map(lambda line: [int(line.split(",")[0]), int(line.split(",")[1])])
cards_transactions.take(5)

[[224, 10], [836, 100], [81, 5], [809, 50], [786, 25]]

**Aggregate by Key** - Use reduceByKey to sum all transaction amounts for each card ID. Returns (card_id, total_amount) pairs.

In [31]:
cards_count = cards_transactions.reduceByKey(lambda a, b: int(a) + int(b))
cards_count.take(5)

[(224, 2556855),
 (592, 2559365),
 (832, 2576415),
 (400, 2552220),
 (128, 2574275)]

**Save Results** - Write the final RDD to a text file. Uses coalesce(1) to output a single partition/file.

In [33]:
cards_count.coalesce(1).saveAsTextFile("out.csv")